# MMS FACEBOOK 

In [1]:
from transformers import AutoTokenizer, VitsModel
import torch
import sounddevice as sd
import time
from torch.nn.utils import prune

c:\Users\kimbe\Documents\GitHub\kata-ondevice\kata-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- TTS ---

mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

def mms_test(text: str):
    inputs = mms_tokenizer(text, return_tensors="pt")
    
    start_time = time.time()
    
    with torch.no_grad():
        output = mms(**inputs).waveform
    
    end_time = time.time()
    
    inference_time = end_time - start_time
    print(f"Inference Time: {inference_time:.4f} seconds")
    
    audio_array = output.squeeze().cpu().numpy()
    sample_rate = 16000 

    sd.play(audio_array, sample_rate)
    sd.wait()

In [3]:
mms_test("Cuaca di Jakarta biasanya panas dan kering.")

Inference Time: 1.9104 seconds


## MMS FACEBOOK QUANTIZED

In [7]:
# Load Model and Tokenizer
mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

# Prune Model Layers
def prune_model_layer(layer, amount=0.2):
    prune.l1_unstructured(layer, name="weight", amount=amount)

for name, module in mms.named_modules():
    if isinstance(module, torch.nn.Linear):  
        prune_model_layer(module, amount=0.2)

for module in mms.modules():
    if isinstance(module, torch.nn.Linear):
        prune.remove(module, "weight")

# Quantize the Pruned Model
quantized_mms = torch.quantization.quantize_dynamic(
    mms,  
    {torch.nn.Linear}, 
    dtype=torch.qint8
)

# Save the Quantized and Pruned Model
torch.save(quantized_mms.state_dict(), "mms_qp.pth")

In [8]:
quantized_mms.load_state_dict(torch.load("mms_qp.pth"))

C:\Users\kimbe\AppData\Local\Temp\ipykernel_253460\1430032170.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  quantized_mms.load_state_dict(torch.load("mms_qp.pth"))


<All keys matched successfully>

In [9]:
mms_test("Cuaca di Jakarta biasanya panas dan kering.")

Inference Time: 1.3068 seconds
